# OT-LoRA-Merge — Kaggle T4 Free-Tier Runner

Canonical reproduction notebook for the 8-task CLIP-ViT-B/32 LoRA benchmark.  
**FREE tier only** (T4/P100, 30 GPU-hr/wk). No ViT-L or Llama-8B. Spec §5/§8.

Cell order:
1. Install pinned stack (FusionBench from source, torch, PEFT, open_clip)
2. Clone repo + `pip install -e .`
3. M0: reproduce baselines (simple-avg, TA, TIES, AdaMerging, TaskSingularVector, ISO-C)
4. M1: run OT-LoRA-Merge (align + barycenter)
5. Aggregate results → README table

Seed: 2026 (fixed throughout). Spec §1.7 determinism.

In [ ]:
# ── Cell 1: Install pinned stack ──────────────────────────────────────────────
# All FREE-tier packages. No paid compute, no gated models. Spec §8.
#
# FusionBench installed from HEAD so we get the latest KnOTS/Core-Space (ISO-C) support.
# VERIFY-IN-KAGGLE: if a specific commit hash is needed for reproducibility, pin:
#   pip install git+https://github.com/tanganke/fusion_bench.git@<commit_sha>
# and record the hash in results/adapter_hashes.json.
#
# torch>=2.2 ships with T4 Kaggle images; we re-pin here for clarity.
# POT 0.9.6: Sinkhorn + GW + free-support barycenter. All in one. Spec §8.

!pip install -q \
    'torch>=2.2,<2.6' \
    'transformers>=4.44,<4.50' \
    'peft>=0.11,<0.14' \
    'open_clip_torch>=2.24' \
    'POT==0.9.6' \
    'numpy==1.26.4' \
    'omegaconf>=2.3' \
    'hydra-core>=1.3' \
    'lightning>=2.2' \
    'datasets>=2.14' \
    'torchvision>=0.17' \
    'torchmetrics>=1.0' \
    'bidict' \
    'pyyaml'

# Install FusionBench from HEAD (MIT license, free)
!pip install -q git+https://github.com/tanganke/fusion_bench.git

# Quick sanity check
import fusion_bench, ot, torch, peft
print(f'fusion_bench={fusion_bench.__version__}  POT={ot.__version__}  torch={torch.__version__}  peft={peft.__version__}')

In [ ]:
# ── Cell 2: Clone repo + pip install -e . ────────────────────────────────────
# Clone the OT-LoRA-Merge repo and install in editable mode so src/ is on sys.path.
# VERIFY-IN-KAGGLE: replace <YOUR_GITHUB_USERNAME> with the actual repo URL once pushed.

import os, subprocess, sys

REPO_URL = 'https://github.com/<YOUR_GITHUB_USERNAME>/ot-lora-merge.git'  # FILL IN
REPO_DIR = '/kaggle/working/ot-lora-merge'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
!pip install -q -e '.[eval]'

# Confirm OT core tests pass (12/12 expected; no GPU needed for these)
!python -m pytest tests -q --tb=short

# Confirm FusionBench wiring imports
from ot_lora_merge.fusionbench_hook import OTLoRAMergeAlgorithm, register
register()
print('OTLoRAMergeAlgorithm registered. Ready.')

In [ ]:
# ── Cell 3: M0 — Baselines ───────────────────────────────────────────────────
# Reproduce published baseline numbers on the 8-task ViT-B/32 LoRA pool.
# Merges run on CPU (milliseconds); eval on T4 (~20 min for all baselines).
# Results written to results/m0_baselines.csv.
#
# VERIFY-IN-KAGGLE: check adapter HF repos exist:
#   hoffman-lab/KnOTS-ViT-B-32_lora_R16_<task>
# If only lora-8 or lora-32 is available, update configs/modelpool/clip_vit_b32_knots_8task_lora.yaml.

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!python experiments/m0_baselines/run.py \
    --config configs/modelpool/clip_vit_b32_knots_8task_lora.yaml \
    --out results/m0_baselines.csv \
    --seed 2026

import csv
print('\n--- M0 Baselines ---')
with open('results/m0_baselines.csv') as f:
    for row in csv.DictReader(f):
        print(f"{row['method']:20s}  avg_norm_acc={float(row['avg_norm_acc'])*100:.2f}%")

In [ ]:
# ── Cell 4: M1 — OT-LoRA-Merge (align + barycenter) ──────────────────────────
# OT solve is CPU (r×r Sinkhorn, milliseconds). Eval on T4.
# Results written to results/m1_ot_align.csv and results/m1_ot_barycenter.csv.

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# M1-align
!python experiments/m1_ot_v1/run.py \
    --config configs/modelpool/clip_vit_b32_knots_8task_lora.yaml \
    --method configs/method/ot_lora_align.yaml \
    --out results/m1_ot_align.csv

# M1-barycenter (headline method)
!python experiments/m1_ot_v1/run.py \
    --config configs/modelpool/clip_vit_b32_knots_8task_lora.yaml \
    --method configs/method/ot_lora_barycenter.yaml \
    --out results/m1_ot_barycenter.csv

import csv, glob
print('\n--- M1 OT-LoRA-Merge ---')
for path in sorted(glob.glob('results/m1_*.csv')):
    with open(path) as f:
        for row in csv.DictReader(f):
            print(f"{row['method']:30s}  avg_norm_acc={float(row['avg_norm_acc'])*100:.2f}%")

In [ ]:
# ── Cell 5: Aggregate results → README table ──────────────────────────────────
# Collects all results/m*.csv and prints a sorted markdown table.
# No GPU needed.

!python scripts/make_results_table.py --glob 'results/*.csv'

# Also print a CSV of all results for reference
import csv, glob
print('\n--- Full results (sorted by avg-norm-acc desc) ---')
all_rows = []
for path in sorted(glob.glob('results/*.csv')):
    with open(path) as f:
        for row in csv.DictReader(f):
            if 'method' in row and 'avg_norm_acc' in row:
                all_rows.append((row['method'], float(row['avg_norm_acc'])))
all_rows.sort(key=lambda x: -x[1])
for method, acc in all_rows:
    print(f'  {method:30s}  {acc*100:.2f}%')